### Scraping from [poets.org](https://www.poets.org/poems)

In [1]:
# Install required packages (run once)
# !pip install requests beautifulsoup4 pandas tqdm

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
import re

In [3]:
base_url = "https://poets.org/poems"

all_data = []

TOTAL_PAGES = 1000

for page in tqdm(range(TOTAL_PAGES), desc="Scraping listing pages"):
    
    url = f"{base_url}?page={page}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception:
        continue
    
    soup = BeautifulSoup(response.text, "html.parser")
    rows = soup.find_all("tr")
    
    for row in rows:
        cols = row.find_all("td")
        if len(cols) < 3:
            continue
        
        a = cols[0].find("a")
        if not a:
            continue
        
        title = a.get_text(strip=True)
        link = "https://poets.org" + a["href"]
        poet = cols[1].get_text(strip=True)
        year = cols[2].get_text(strip=True)
        
        all_data.append({
            "Title": title,
            "Poet": poet,
            "Year": year,
            "URL": link
        })
    
    time.sleep(0.1)

df = pd.DataFrame(all_data)
print("Metadata rows:", len(df))

Scraping listing pages: 100%|██████████| 1000/1000 [21:56<00:00,  1.32s/it]

Metadata rows: 16713


In [4]:
def clean_text(text):
    if not text:
        return None
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def get_poem_text(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        container = soup.find("div", class_="field--body")
        if not container:
            return None

        # remove junk elements
        for tag in container.find_all(["img", "blockquote"]):
            tag.decompose()

        # structured spans (most poems)
        spans = container.find_all("span")
        lines = []

        if spans:
            for span in spans:

                # remove hidden erasure text
                for hidden in span.find_all(style=lambda v: v and "color:#FFFFFF" in v):
                    hidden.decompose()

                text = clean_text(span.get_text(" ", strip=True))

                if text:
                    lines.append(text)

            if lines:
                return "\n".join(lines)

        # preformatted poems
        pre = container.find("pre")
        if pre:
            return clean_text(pre.get_text("\n", strip=True))

        # fallback
        return clean_text(container.get_text("\n", strip=True))

    except Exception:
        return None

In [5]:
poems = []

checkpoint_interval = 500

for i, url in enumerate(tqdm(df["URL"], desc="Scraping poems")):
    
    poems.append(get_poem_text(url))
    
    # checkpoint
    if i % checkpoint_interval == 0 and i > 0:
        df["Poem Text"] = poems + [None] * (len(df) - len(poems))
        df.to_csv("poets_checkpoint.csv", index=False)
        print(f"Checkpoint saved at {i}")
    
    time.sleep(0.1)

df["Poem Text"] = poems

Scraping poems:   3%|▎         | 501/16713 [07:59<4:39:41,  1.04s/it]

Checkpoint saved at 500


Scraping poems:   6%|▌         | 1001/16713 [16:48<4:38:08,  1.06s/it]

Checkpoint saved at 1000


Scraping poems:   9%|▉         | 1501/16713 [25:19<4:37:08,  1.09s/it]

...

Scraping poems:  96%|█████████▌| 16001/16713 [4:19:42<09:40,  1.23it/s]

Checkpoint saved at 16000


Scraping poems:  99%|█████████▊| 16501/16713 [4:26:44<03:09,  1.12it/s]

Checkpoint saved at 16500


Scraping poems: 100%|██████████| 16713/16713 [4:29:41<00:00,  1.03it/s]


In [6]:
df.to_csv("poets_full_dataset.csv", index=False)
print("DONE — saved poets_full_dataset.csv")

DONE — saved poets_full_dataset.csv
